In [1]:
# NORTHSTAR -- Tower 46 Mobile Apps QA for Solace Browser
# DNA: mobile(ios) = pwa(installable) x capacitor(native) x appstore(distribution) x offline(resilient) x notification(push) x auth(faceid) x responsive(adaptive)
# Probes responsive design, PWA support, Capacitor config, mobile CI, touch targets
import os, sys, re, json, hashlib
from pathlib import Path
from datetime import datetime

NORTHSTAR = "mobile-apps-t46-qa"
NOTEBOOK_PATH = "notebooks/qa/62-mobile-apps-t46-qa.ipynb"
TOWER = 46
TOWER_NAME = "Tower of Mobile Apps"
MIN_SCORE = 70

BROWSER_ROOT = Path("/home/phuc/projects/solace-browser")
SRC = BROWSER_ROOT / "src"
WEB = BROWSER_ROOT / "web"
TESTS = BROWSER_ROOT / "tests"
DATA = BROWSER_ROOT / "data" / "default"

results = []

def record(probe_id, name, passed, detail=""):
    status = "PASS" if passed else "FAIL"
    results.append({"id": probe_id, "name": name, "status": status, "detail": detail})
    print(f"{status}: {name}" + (f" -- {detail}" if detail else ""))

def read_file(path):
    p = Path(path)
    return p.read_text(encoding='utf-8', errors='replace') if p.exists() else ""

print(f"Tower {TOWER}: {TOWER_NAME}")
assert BROWSER_ROOT.exists()

Tower 46: Tower of Mobile Apps


In [2]:
# F1 -- Emily Richardson: iPhone User Experience
# PWA manifest, service worker, viewport meta tags

# Probe: manifest.json exists with valid PWA fields
manifest_raw = read_file(WEB / "manifest.json")
has_manifest = len(manifest_raw) > 50
record("F1-P1", "PWA manifest.json exists", has_manifest,
       f"{len(manifest_raw)} bytes")

if has_manifest:
    manifest = json.loads(manifest_raw)
    has_name = "name" in manifest and len(manifest["name"]) > 0
    record("F1-P2", "Manifest has app name", has_name,
           manifest.get("name", ""))
    has_display = manifest.get("display") == "standalone"
    record("F1-P3", "Manifest display=standalone (app-like)", has_display,
           f"display={manifest.get('display', 'missing')}")
    icons = manifest.get("icons", [])
    record("F1-P4", "Manifest has icons (>=3)", len(icons) >= 3,
           f"{len(icons)} icons")
else:
    record("F1-P2", "Manifest has app name", False, "manifest missing")
    record("F1-P3", "Manifest display=standalone", False, "manifest missing")
    record("F1-P4", "Manifest has icons", False, "manifest missing")

# Probe: Service worker exists
sw_code = read_file(WEB / "sw.js")
has_sw = bool(re.search(r'addEventListener.*install|cache|fetch', sw_code, re.IGNORECASE))
record("F1-P5", "Service worker (sw.js) exists with install handler", has_sw,
       f"{len(sw_code)} bytes")

PASS: PWA manifest.json exists -- 2327 bytes
PASS: Manifest has app name -- Solace Browser
PASS: Manifest display=standalone (app-like) -- display=standalone
PASS: Manifest has icons (>=3) -- 6 icons
PASS: Service worker (sw.js) exists with install handler -- 3783 bytes


In [3]:
# F2 -- Kenji Nakamura: Capacitor / Native Bridge
# Capacitor config, android directory, build workflow

# Probe: capacitor.config.json exists
cap_config_raw = read_file(BROWSER_ROOT / "capacitor.config.json")
has_cap = len(cap_config_raw) > 20
record("F2-P1", "capacitor.config.json exists", has_cap,
       f"{len(cap_config_raw)} bytes")

if has_cap:
    cap_config = json.loads(cap_config_raw)
    has_appid = "appId" in cap_config
    record("F2-P2", "Capacitor config has appId", has_appid,
           cap_config.get("appId", "missing"))
    has_ios_config = "ios" in cap_config
    record("F2-P3", "Capacitor config has iOS section", has_ios_config)
else:
    record("F2-P2", "Capacitor config has appId", False, "config missing")
    record("F2-P3", "Capacitor config has iOS section", False, "config missing")

# Probe: Android directory exists (Capacitor generated)
android_dir = BROWSER_ROOT / "android"
has_android = android_dir.exists() and (android_dir / "app").exists()
record("F2-P4", "Android Capacitor directory exists", has_android)

# Probe: build-mobile.yml CI workflow exists
mobile_ci = read_file(BROWSER_ROOT / ".github" / "workflows" / "build-mobile.yml")
has_mobile_ci = bool(re.search(r'cap.*sync|capacitor|android|gradle', mobile_ci, re.IGNORECASE))
record("F2-P5", "build-mobile.yml CI workflow exists", has_mobile_ci,
       f"{len(mobile_ci)} bytes")

PASS: capacitor.config.json exists -- 580 bytes
PASS: Capacitor config has appId -- com.solaceagi.browser
PASS: Capacitor config has iOS section
PASS: Android Capacitor directory exists
PASS: build-mobile.yml CI workflow exists -- 2593 bytes


In [4]:
# F3 -- Margaret O'Brien: Responsive Design (iPad/tablet)
# Viewport meta, media queries, mobile-first CSS

# Probe: Viewport meta tag on HTML pages
html_files = list(WEB.glob("*.html"))
viewport_count = 0
for f in html_files:
    content = f.read_text(encoding='utf-8', errors='replace')
    if re.search(r'viewport.*width=device-width', content):
        viewport_count += 1
record("F3-P1", "Viewport meta on pages (>=10)",
       viewport_count >= 10, f"{viewport_count}/{len(html_files)} pages")

# Probe: Responsive media queries in site.css
site_css = read_file(WEB / "css" / "site.css")
media_queries = re.findall(r'@media.*max-width', site_css)
record("F3-P2", "Responsive media queries (>=5)",
       len(media_queries) >= 5, f"{len(media_queries)} @media rules")

# Probe: Mobile menu CSS exists
has_mobile_menu = bool(re.search(r'\.mobile-menu', site_css))
record("F3-P3", "Mobile menu CSS exists", has_mobile_menu)

PASS: Viewport meta on pages (>=10) -- 17/20 pages
PASS: Responsive media queries (>=5) -- 15 @media rules
PASS: Mobile menu CSS exists


In [5]:
# F4/F5 -- Touch Targets + Performance
# Touch-friendly CSS (min 44px targets), lazy loading

# Probe: Touch-friendly target sizes (min-height 44px or 48px)
touch_targets = re.findall(r'min-(?:height|width):\s*(?:44|48)px', site_css)
record("F4-P1", "Touch-friendly target sizes (>=3 rules)",
       len(touch_targets) >= 3, f"{len(touch_targets)} touch target rules")

# Probe: Package.json has Capacitor dependencies
pkg_json_raw = read_file(BROWSER_ROOT / "package.json")
has_pkg = len(pkg_json_raw) > 50
if has_pkg:
    pkg_json = json.loads(pkg_json_raw)
    all_deps = {}
    all_deps.update(pkg_json.get("dependencies", {}))
    all_deps.update(pkg_json.get("devDependencies", {}))
    cap_deps = [k for k in all_deps if "capacitor" in k.lower()]
    record("F5-P1", "Capacitor dependencies in package.json",
           len(cap_deps) >= 1, f"{len(cap_deps)} capacitor deps: {cap_deps[:5]}")
else:
    record("F5-P1", "Capacitor dependencies in package.json", False, "package.json missing")

# F6 -- Accessibility on Mobile
# Probe: prefers-reduced-motion in CSS
has_reduced_motion = bool(re.search(r'prefers-reduced-motion', site_css))
record("F6-P1", "prefers-reduced-motion CSS support", has_reduced_motion)

# Probe: prefers-contrast in CSS
has_contrast = bool(re.search(r'prefers-contrast', site_css))
record("F6-P2", "prefers-contrast CSS support", has_contrast)

PASS: Touch-friendly target sizes (>=3 rules) -- 16 touch target rules
PASS: Capacitor dependencies in package.json -- 4 capacitor deps: ['@capacitor/android', '@capacitor/cli', '@capacitor/core', '@capacitor/ios']
PASS: prefers-reduced-motion CSS support
PASS: prefers-contrast CSS support


In [6]:
# F7 -- Mei-Ling Wu: App Store Distribution
# Related applications, screenshots, distribution test

if has_manifest:
    manifest = json.loads(manifest_raw)
    # Probe: Related applications (Google Play link)
    related = manifest.get("related_applications", [])
    has_related = len(related) > 0
    record("F7-P1", "Manifest lists related_applications", has_related,
           f"{len(related)} related apps")
    # Probe: Screenshots for app store listing
    screenshots = manifest.get("screenshots", [])
    has_screenshots = len(screenshots) > 0
    record("F7-P2", "Manifest has screenshots for store listing", has_screenshots,
           f"{len(screenshots)} screenshots")
    # Probe: Shortcuts defined
    shortcuts = manifest.get("shortcuts", [])
    has_shortcuts = len(shortcuts) > 0
    record("F7-P3", "Manifest has app shortcuts", has_shortcuts,
           f"{len(shortcuts)} shortcuts")
else:
    record("F7-P1", "Manifest lists related_applications", False, "manifest missing")
    record("F7-P2", "Manifest has screenshots", False, "manifest missing")
    record("F7-P3", "Manifest has app shortcuts", False, "manifest missing")

# Probe: Distribution test exists
dist_test = TESTS / "test_distribution.py"
has_dist_test = dist_test.exists() and dist_test.stat().st_size > 100
record("F7-P4", "Distribution test exists", has_dist_test)

PASS: Manifest lists related_applications -- 1 related apps
PASS: Manifest has screenshots for store listing -- 2 screenshots
PASS: Manifest has app shortcuts -- 2 shortcuts
PASS: Distribution test exists


In [7]:
# EVIDENCE SUMMARY -- Tower 46 Mobile Apps QA
passed = len([r for r in results if r["status"] == "PASS"])
failed = len([r for r in results if r["status"] == "FAIL"])
total = len(results)
score = round(passed / total * 100, 1) if total > 0 else 0
finding_rate = round(failed / total * 100, 1) if total > 0 else 0
grade = "A+" if score >= 95 else "A" if score >= 90 else "B+" if score >= 85 else "B" if score >= 80 else "C" if score >= 70 else "D" if score >= 60 else "F"

summary = {
    "surface": NORTHSTAR, "tower": TOWER, "tower_name": TOWER_NAME,
    "timestamp": datetime.now().isoformat(),
    "total_probes": total, "passed": passed, "failed": failed,
    "score": score, "grade": grade, "finding_rate": finding_rate,
    "converged": finding_rate < 5,
    "findings": [r for r in results if r["status"] == "FAIL"],
    "evidence_hash": hashlib.sha256(json.dumps(results, sort_keys=True).encode()).hexdigest()[:16]
}

print("=" * 60)
print(f"TOWER {TOWER}: {TOWER_NAME}")
print("=" * 60)
print(f"Probes: {total} | Passed: {passed} | Failed: {failed}")
print(f"Score: {score}% | Grade: {grade} | Finding rate: {finding_rate}%")
print(f"Converged: {'YES' if summary['converged'] else 'NO'}")
print(f"Evidence hash: {summary['evidence_hash']}")
if summary["findings"]:
    print("\nFINDINGS:")
    for f in summary["findings"]:
        print(f"  [{f['id']}] {f['name']}: {f.get('detail', '')}")
print()
print(json.dumps(summary, indent=2))

TOWER 46: Tower of Mobile Apps
Probes: 21 | Passed: 21 | Failed: 0
Score: 100.0% | Grade: A+ | Finding rate: 0.0%
Converged: YES
Evidence hash: 9fe8bdfb99bc3712

{
  "surface": "mobile-apps-t46-qa",
  "tower": 46,
  "tower_name": "Tower of Mobile Apps",
  "timestamp": "2026-03-06T11:30:01.370296",
  "total_probes": 21,
  "passed": 21,
  "failed": 0,
  "score": 100.0,
  "grade": "A+",
  "finding_rate": 0.0,
  "converged": true,
  "findings": [],
  "evidence_hash": "9fe8bdfb99bc3712"
}
